# Acciones iniciales

In [ ]:
from google.colab import drive # montar drive
import shutil # copiar docs. drive en disco local
from pathlib import Path # Gestionar rutas archivos
import csv # Ver formato csv
from charset_normalizer import from_path # Ver codificacion archivos
import codecs # Para ver distintos nombre de un mismo tipo de codificacion
import json # Para crear y tratar archivo json
import pandas as pd

In [ ]:
# Montar drive:
#drive.mount('/content/drive')

# Constantes

In [ ]:
# Para tratar archivos

# Rutas en Drive para cada caso particular
DRIVE_ORIGEN = drive / '1_BRONZE'

# Rutas en local, con los que se trabaja en la sesion
ORIGEN_CSV = local / 'tablas_bronze'
DESTINO_CSV = local / 'resultados_codificacion_finales'
DESTINO_CSV.mkdir(parents=True, exist_ok=True)   # línea aparte. Crea resulados_codificacion de no exisitir y carpetas intermedias si surgen

# Para la funcion
DELIMITADORES = ",;\t|"
TAM_MUESTRA = 64 * 1024  # en caracteres
COLS_FORMATO = ["encoding", "sep", "quotechar"]
CORRECCIONES_ENCODING = {"cp1250": "cp1252",
                         "cp1257": "cp1252",
                         "iso8859-2": "cp1252",
                         "iso8859-1": "cp1252"}

# Copia archivos drive en local

In [ ]:
# Copiar Drive -> local

assert DRIVE_ORIGEN.exists(), f"No existe {DRIVE_ORIGEN}"

# Copia DRIVE_ORIGEN completo, con sus subcarpetas (si las hay), a ORIGEN_CSV
shutil.copytree(DRIVE_ORIGEN, ORIGEN_CSV, dirs_exist_ok=True)

DESTINO_CSV.mkdir(parents=True, exist_ok=True)

print(f"CSV copiados a local: {len(list(ORIGEN_CSV.rglob('*.csv')))} archivos .csv\n")

# Los nombres de las subcarpetas se leen de DRIVE_ORIGEN
subcarpetas = sorted(p.name for p in DRIVE_ORIGEN.iterdir() if p.is_dir())

if not subcarpetas:
    print("DRIVE_ORIGEN no tiene subcarpetas.")

for sub in subcarpetas:
    csvs = sorted(p.name for p in (ORIGEN_CSV / sub).rglob("*.csv"))
    print(f"{sub} ({len(csvs)} csv): {csvs}")

# CSV que estén sueltos en la raíz, fuera de cualquier subcarpeta
csvs_raiz = sorted(p.name for p in ORIGEN_CSV.glob("*.csv"))
if csvs_raiz:
    print(f"(raíz) ({len(csvs_raiz)} csv): {csvs_raiz}")

# Fución detectar codificación CSVs

In [ ]:
def detectar_codificacion(ruta: Path) -> dict:
    """Detecta la codificación del archivo con charset_normalizer.

    Devuelve el encoding detectado y su nivel de caos (0-1, menor es mejor).
    Si no se detecta ninguno, encoding es None.
    """
    resultado = from_path(ruta).best()
    if resultado is None:
        return {"encoding": None, "caos": None} #Sino se detecta encoding lo avisa, no pone uno por defecto
    return {"encoding": resultado.encoding, "caos": round(resultado.chaos, 3)}


def normalizar_encoding(encoding: str) -> str:
    """
    Unifica el nombre del encoding y usa utf-8-sig para UTF-8 y ASCII.
    """
    nombre = codecs.lookup(encoding).name  # 'utf_8' -> 'utf-8'
    # codecs.lookup busca en el directorio de codificaciones y muestra la estandar para el grupo.
    # una misma codificacion puede escribirse de maneras distintas.

   # Detecta el error del cp1250 centroeuropeas en codificaciones cp1252 españolas
    nombre = CORRECCIONES_ENCODING.get(nombre, nombre)
    return "utf-8-sig" if nombre in ("utf-8", "ascii") else nombre
     #retorna utf-8-sig solo para utf-8, etc porque es mas segura de leer.


def detectar_formato(ruta: Path, encoding: str) -> dict:
    """
    Detecta separador y quotechar a partir de una muestra del archivo.
    """
    with ruta.open("r", encoding=encoding, newline="") as f:
        muestra = f.read(TAM_MUESTRA)
    if len(muestra) == TAM_MUESTRA:  # muestra cortada: quitar la última línea incompleta
        muestra = muestra[: muestra.rfind("\n") + 1] or muestra
    dialecto = csv.Sniffer().sniff(muestra, delimiters=DELIMITADORES)
    return {"sep": dialecto.delimiter, "quotechar": dialecto.quotechar}


def union_informacion(carpeta: Path) -> dict:
    """
    Analiza todos los CSV de la carpeta y subcarpetas.
    Devuelve un diccionario {ruta_relativa: datos}. Los archivos que fallan
    se registran con su error en lugar de detener el proceso.
    """
    resultados = {}
    for archivo in sorted(carpeta.rglob("*.csv")):
        rel = archivo.relative_to(carpeta).as_posix() #nombre de la ruta
        try:
            cod = detectar_codificacion(archivo)
            if cod["encoding"] is None:
                raise ValueError("no se pudo detectar la codificación")
            encoding = normalizar_encoding(cod["encoding"])
            formato = detectar_formato(archivo, encoding)
            resultados[rel] = {
                "read_kwargs": {"encoding": encoding, **formato}, #lo que se leera como parametro al abrir csv
                "encoding_detectado": cod["encoding"], # Datos de resumen
                "caos": cod["caos"],
                "error": None,
            }
        except (UnicodeDecodeError, LookupError, csv.Error, ValueError, OSError) as e:
            resultados[rel] = {"read_kwargs": None, "error": f"{type(e).__name__}: {e}"}
    return resultados


def construir_df(info: dict) -> pd.DataFrame:
    """
    Convierte el diccionario de parámetros en un DataFrame con una fila por archivo.
    """
    filas = [
        {
            "archivo": Path(rel).name,
            "carpeta": Path(rel).parent.as_posix(),
            **(d.get("read_kwargs") or {}),
            "caos": d.get("caos"),
            "encoding_detectado": d.get("encoding_detectado"),
            "error": d.get("error"),
        }
        for rel, d in info.items()
    ]
    df = pd.DataFrame(filas, columns=["carpeta",
                                      "archivo",
                                      *COLS_FORMATO,
                                      "encoding_detectado",
                                      "caos", "error"])
    df["carpeta"] = df["carpeta"].replace(".", "(raíz)")
    return df.set_index(["carpeta", "archivo"])


def mostrar_resumen(df: pd.DataFrame, titulo: str) -> None:

    """
    Imprime el resumen de un grupo de archivos y avisa si no es homogéneo.
    """

    n = len(df)
    discrepantes = df["error"].notna()  # los que fallaron se muestran siempre
    empates = []
    errores = df["error"].notna().sum()
    corregidos = df[df["encoding_detectado"].notna() &
     (df["encoding"] != "utf-8-sig") &
      (df["encoding_detectado"] != df["encoding"])]
    print(f"\n------ {titulo}: {n} archivos, {errores} con error -----")

    if titulo == "GENERAL": #Solo se imprime resumen valores
        for col in COLS_FORMATO:
          conteo = df[col].value_counts(dropna=False)
          texto = ", ".join(f"{valor!r}: {c} ({c / n:.0%})" for valor, c in conteo.items())
          print(f"  {col}: {texto}")
    else:
        for col in COLS_FORMATO:
            conteo = df[col].value_counts(dropna=False)
            texto = ", ".join(f"{valor!r}: {c} ({c / n:.0%})" for valor, c in conteo.items())
            print(f"  {col}: {texto}")
            if conteo.empty:
                continue
            if len(conteo) > 1 and conteo.iloc[0] == conteo.iloc[1]:
                empates.append(col)  # no hay valor mayoritario: no se compara
                continue
            discrepantes |= df[col] != conteo.index[0]

        if empates:
            print(f"\n  Sin valor mayoritario (empate) en: {', '.join(empates)}. Revisar el reparto.")

        if discrepantes.any():
            print("\n  Archivos que no coinciden con el valor mayoritario:\n")
            print(df.loc[discrepantes, [*COLS_FORMATO]].to_string())

        if not corregidos.empty:
            print("\n  Encodings corregidos:\n")
            for (carpeta, archivo), fila in corregidos.iterrows():
                print(f"{archivo}: Encoding detectado: {fila['encoding_detectado']} → encoding correcto: {fila['encoding']}")


def mostrar_resumenes(info: dict) -> pd.DataFrame:

    """
    Muestra el resumen general y uno por subcarpeta. Devuelve el DataFrame.
    """

    df = construir_df(info)
    mostrar_resumen(df, "GENERAL")
    for carpeta, grupo in df.groupby("carpeta"):
        mostrar_resumen(grupo, carpeta)
    return df


def revision_inicial(carpeta: Path, carpeta_json: Path):
    """
    Detecta formato de todos los CSV, guarda el JSON y muestra un resumen.
    Atención: sobrescribe el JSON, incluidas las correcciones manuales.
    retorna la ruta del archivo generado y el df_resmen de los resultados obtenidos
    """
    arc_json = carpeta_json / 'parametros_csv_brutos.json'

    info = union_informacion(carpeta)

    with arc_json.open("w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False) # Se crea JSON

    return arc_json, mostrar_resumenes(info)

# Copia archivo locales en drive

In [ ]:
# local -> Drive
# Para subir solo el archivo JSON generado a la misma carpeta de DRIVE_ORIGEN:

if archivo_json.exists(): # el nombre del archivo a subir se retorna en la funcion revision_inicial
    shutil.copy2(archivo_json, DRIVE_ORIGEN)
    print('Subido:', DRIVE_ORIGEN / archivo_json.name)
else:
    print('AVISO: no existe', archivo_generado, '- no se copia nada a Drive')

drive.flush_and_unmount()